# Run Full Pipeline

This notebook runs the canonical pipeline end-to-end using the shared settings from `src/notebooks/shared_config.json`.

Stages covered:

- scan
- extract documents
- extract events
- filter midnight events
- pair employee events
- enrich shifts
- build yearly summary
- audit missing timbrature


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.notebooks.pipeline_config import load_notebook_context

ctx = load_notebook_context()
paths = ctx.paths
scan_cfg = ctx.step("scan")
extract_cfg = ctx.step("extract_documents")
events_cfg = ctx.step("extract_events")
filter_cfg = ctx.step("filter_midnight")
pair_cfg = ctx.step("pair_employee")
enrichment_cfg = ctx.step("turni_enrichment")
summary_cfg = ctx.step("turni_employee_summary")
audit_cfg = ctx.step("timbrature_missing_report")

{
    "config_path": str(ctx.config_path),
    "root_id": ctx.root_id,
    "root_output": str(paths.root_output),
}


{'config_path': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\src\\notebooks\\shared_config.json',
 'root_id': '1T6w6Np9WaXUoFWnjoFadiMv_I4xe7K4t',
 'root_prefix': 'GASPERINI',
 'root_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI'}

In [5]:
VERBOSE = True
REPROCESS_INCLUDED = False
REPROCESS_EXCLUDED = False

scan_included_path = paths.scan_output / scan_cfg["included_name"]
scan_filtered_path = paths.scan_output / scan_cfg["filtered_name"]
scan_report_path = paths.scan_output / scan_cfg["report_name"]

extract_report_path = paths.documents_output / extract_cfg["report_name"]
events_report_path = paths.events_output / events_cfg["report_name"]
filter_report_path = paths.events_output / filter_cfg["report_name"]
removed_csv_path = paths.events_output / filter_cfg["removed_csv_name"]
pair_report_path = paths.shifts_output / pair_cfg["report_name"]
enrichment_report_path = paths.enrichment_output / enrichment_cfg["report_name"]
summary_output_path = paths.aggregation_output / summary_cfg["out_name"]
summary_report_path = paths.aggregation_output / summary_cfg["report_name"]
audit_report_path = paths.root_output / audit_cfg["report_name"]
audit_summary_path = paths.root_output / audit_cfg["summary_name"]
audit_findings_path = paths.root_output / audit_cfg["findings_name"]
audit_coverage_path = paths.root_output / audit_cfg["coverage_name"]

{
    "scan_output": str(paths.scan_output),
    "documents_output": str(paths.documents_output),
    "events_output": str(paths.events_output),
    "shifts_output": str(paths.shifts_output),
    "enrichment_output": str(paths.enrichment_output),
    "aggregation_output": str(paths.aggregation_output),
    "audit_report": str(audit_report_path),
}


{'scan_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\scan',
 'documents_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\documents',
 'events_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\events',
 'shifts_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\shifts',
 'enrichment_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\enrichment',
 'aggregation_output': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\aggregation',
 'audit_report': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\missing_timbrature.report.json'}

In [ ]:
from src.drive_service.auth_service import load_creds
from src.drive_service.drive_client import get_drive_service
from src.drive_service.logging_utils import setup_logging
from src.scan_directory.runtime import run_scan

setup_logging(VERBOSE)

creds = load_creds()
drive = get_drive_service(creds)

scan_report = run_scan(
    creds=creds,
    drive=drive,
    root_id=ctx.root_id,
    workers=int(scan_cfg["workers"]),
    included_path=str(scan_included_path),
    filtered_path=str(scan_filtered_path),
    report_path=str(scan_report_path),
)
scan_report


In [ ]:
import json

from src.extract_documents_from_index.options import ExtractDocumentsFromIndexOptions
from src.extract_documents_from_index.runtime import run_extraction

extract_options = ExtractDocumentsFromIndexOptions(
    out=str(paths.documents_output),
    index=str(scan_included_path),
    included=str(extract_cfg["included_name"]),
    excluded=str(extract_cfg["excluded_name"]),
    report=str(extract_cfg["report_name"]),
    reprocess_included=REPROCESS_INCLUDED,
    reprocess_excluded=REPROCESS_EXCLUDED,
    verbose=VERBOSE,
)

extract_exit_code = run_extraction(extract_options)
extract_report = json.loads(extract_report_path.read_text(encoding="utf-8"))

{
    "exit_code": extract_exit_code,
    "report_json": str(extract_report_path),
    "stats": extract_report["stats"],
}


In [ ]:
from src.extract_events_from_documents.options import ExtractEventsFromTextOptions
from src.extract_events_from_documents.service import run_from_options as run_extract_events

extract_events_options = ExtractEventsFromTextOptions(
    input_dir=str(paths.documents_output),
    output_dir=str(paths.events_output),
    out_name=str(events_cfg["out_name"]),
    pages_name=str(events_cfg["pages_name"]),
    report_json=str(events_report_path),
    verbose=VERBOSE,
)

events_report = run_extract_events(extract_events_options)

{
    "report_json": str(events_report_path),
    "files_ok": events_report.get("files_ok"),
    "files_failed": events_report.get("files_failed"),
}


In [ ]:
from src.filter_midnight_events.options import FilterMidnightEventsOptions
from src.filter_midnight_events.service import run_from_options as run_filter_midnight

filter_options = FilterMidnightEventsOptions(
    input_dir=str(paths.events_output),
    events_name=str(filter_cfg["events_name"]),
    out_name=str(filter_cfg["out_name"]),
    report_json=str(filter_report_path),
    removed_csv=str(removed_csv_path),
    verbose=VERBOSE,
)

filter_report = run_filter_midnight(filter_options)

{
    "report_json": str(filter_report_path),
    "removed_rows_csv": str(removed_csv_path),
    "files_processed": filter_report.get("files_processed"),
    "rows_removed": filter_report.get("rows_removed"),
}


In [ ]:
from src.pair_employee_events.options import PairEmployeeEventsOptions
from src.pair_employee_events.runtime import run_from_options as run_pair_employee

pair_options = PairEmployeeEventsOptions(
    input_dir=str(paths.events_output),
    output_dir=str(paths.shifts_output),
    events_name=str(pair_cfg["events_name"]),
    report_json=str(pair_report_path),
    max_gap_hours=float(pair_cfg["max_gap_hours"]),
    verbose=VERBOSE,
)

pair_report = run_pair_employee(pair_options)

{
    "report_json": str(pair_report_path),
    "employees_processed": pair_report.get("employees_processed"),
    "pairs_written": pair_report.get("pairs_written"),
}


In [ ]:
from src.turni_enrichment.options import TurniEnrichmentOptions
from src.turni_enrichment.service import run_from_options as run_turni_enrichment

enrichment_options = TurniEnrichmentOptions(
    input_dir=str(paths.shifts_output),
    output_dir=str(paths.enrichment_output),
    min_hours=float(enrichment_cfg["min_hours"]),
    include_holidays=bool(enrichment_cfg["include_holidays"]),
    report_json=str(enrichment_report_path),
    verbose=VERBOSE,
)

enrichment_report = run_turni_enrichment(enrichment_options)

{
    "report_json": str(enrichment_report_path),
    "stats": enrichment_report.get("stats"),
}


In [ ]:
from src.turni_employee_summary.options import TurniEmployeeSummaryOptions
from src.turni_employee_summary.service import run_from_options as run_turni_summary

summary_min_hours = summary_cfg.get("min_hours")
summary_options = TurniEmployeeSummaryOptions(
    enriched_dir=str(paths.enrichment_output),
    out=str(summary_output_path),
    report_json=str(summary_report_path),
    output_format=str(summary_cfg["output_format"]),
    year_start=int(summary_cfg["year_start"]),
    year_end=int(summary_cfg["year_end"]),
    min_hours=float(summary_min_hours) if summary_min_hours is not None else None,
    verbose=VERBOSE,
)

summary_report = run_turni_summary(summary_options)

{
    "report_json": str(summary_report_path),
    "output_path": summary_report.get("output_path"),
}


In [6]:
from src.timbrature_missing_report.options import TimbratureMissingReportOptions
from src.timbrature_missing_report.service import run_from_options as run_timbrature_audit

audit_options = TimbratureMissingReportOptions(
    pipeline_dir=str(paths.root_output),
    report_json=str(audit_cfg["report_name"]),
    summary_csv=str(audit_cfg["summary_name"]),
    findings_csv=str(audit_cfg["findings_name"]),
    coverage_csv=str(audit_cfg["coverage_name"]),
    verbose=VERBOSE,
)

audit_report = run_timbrature_audit(audit_options)

{
    "report_json": str(audit_report_path),
    "summary_csv": str(audit_summary_path),
    "findings_csv": str(audit_findings_path),
    "coverage_csv": str(audit_coverage_path),
    "employees": int(audit_report.get("row_totals", {}).get("items", 0)),
    "findings": int(audit_report.get("row_totals", {}).get("issues", 0)),
    "coverage_rows": int(audit_report.get("row_totals", {}).get("coverage_rows", 0)),
}


{'report_json': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\missing_timbrature.report.json',
 'summary_csv': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\missing_timbrature.summary.csv',
 'findings_csv': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\missing_timbrature.findings.csv',
 'coverage_csv': 'C:\\Users\\Rosario\\Desktop\\dev\\Nursind\\output\\GASPERINI\\missing_timbrature.coverage.csv',
 'employees': 21,
 'findings': 24,
 'coverage_rows': 2155}

In [ ]:
{
    "scan_report": str(scan_report_path),
    "extract_report": str(extract_report_path),
    "events_report": str(events_report_path),
    "filter_report": str(filter_report_path),
    "pair_report": str(pair_report_path),
    "enrichment_report": str(enrichment_report_path),
    "summary_report": str(summary_report_path),
    "summary_output": str(summary_output_path),
    "audit_report": str(audit_report_path),
    "audit_summary": str(audit_summary_path),
    "audit_findings": str(audit_findings_path),
    "audit_coverage": str(audit_coverage_path),
}
